<center>
    <h1>JaxTon</h1>
    <i>💯 JAX exercises</i>
    <br>
    <br>
    <a href='https://github.com/vopani/jaxton/blob/master/LICENSE'>
        <img src='https://img.shields.io/badge/license-Apache%202.0-blue.svg?logo=apache'>
    </a>
    <a href='https://github.com/vopani/jaxton'>
        <img src='https://img.shields.io/github/stars/vopani/jaxton?color=yellowgreen&logo=github'>
    </a>
    <a href='https://twitter.com/vopani'>
        <img src='https://img.shields.io/twitter/follow/vopani'>
    </a>
</center>

<center>
    This is Set 9: Neural Networks (Solutions 81-90) of <b>JaxTon</b>: <i>💯 JAX exercises</i>
    <br>
    You can find all the exercises and solutions on <a href="https://github.com/vopani/jaxton#exercises-">GitHub</a>
</center>

**Prerequisites**

* JAX should be installed via `uv sync` in the project root.
* Sample data and helper functions will be used for the exercises.

In [ ]:
import jax
import jax.numpy as jnp

In [ ]:
## sample dataset: XOR problem
X = jnp.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=jnp.float32)
y = jnp.array([[0], [1], [1], [0]], dtype=jnp.float32)

In [ ]:
## helper: ReLU activation
def relu(x):
    return jnp.maximum(0, x)

**Exercise 81: Write a function `init_layer(key, n_in, n_out)` that returns a dict with `"weights"` of shape (n_in, n_out) using He initialization and `"bias"` of zeros. Test with (2, 4).**

In [ ]:
def init_layer(key, n_in, n_out):
    W_key, b_key = jax.random.split(key)
    weights = jax.random.normal(W_key, (n_in, n_out)) * jnp.sqrt(2.0 / n_in)
    bias = jnp.zeros(n_out)
    return {"weights": weights, "bias": bias}

init_layer(jax.random.key(0), 2, 4)

**Exercise 82: Write a function `init_mlp(key, layer_sizes)` that returns a list of layer parameter dicts for an MLP with the given layer sizes (e.g. `[2, 4, 1]`). Assign to `params`.**

In [ ]:
def init_mlp(key, layer_sizes):
    params = []
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        key, subkey = jax.random.split(key)
        params.append(init_layer(subkey, n_in, n_out))
    return params

params = init_mlp(jax.random.key(42), [2, 4, 1])
jax.tree.map(lambda x: x.shape, params)

**Exercise 83: Write a `forward(params, x)` function that performs a forward pass through the MLP. Use `relu` activation for hidden layers and no activation on the output layer. Test with `params` and `X`.**

In [ ]:
def forward(params, x):
    *hidden, last = params
    for layer in hidden:
        x = relu(x @ layer["weights"] + layer["bias"])
    return x @ last["weights"] + last["bias"]

forward(params, X)

**Exercise 84: Write a `mse_loss(params, x, y)` function that computes mean squared error between `forward(params, x)` and `y`. Compute the loss on the XOR dataset.**

In [ ]:
def mse_loss(params, x, y):
    preds = forward(params, x)
    return jnp.mean((preds - y) ** 2)

mse_loss(params, X, y)

**Exercise 85: Use `jax.value_and_grad(mse_loss)` to compute both the loss and gradients of `params` on the XOR data. Assign to `loss_val` and `grads`.**

In [ ]:
loss_val, grads = jax.value_and_grad(mse_loss)(params, X, y)
print(f"Loss: {loss_val}")
jax.tree.map(lambda x: x.shape, grads)

**Exercise 86: Write an `sgd_update(params, grads, lr)` function that uses `jax.tree.map` to subtract `lr * grad` from each parameter. Apply it with lr=0.1.**

In [ ]:
def sgd_update(params, grads, lr):
    return jax.tree.map(lambda p, g: p - lr * g, params, grads)

updated = sgd_update(params, grads, 0.1)
jax.tree.map(lambda x: x.shape, updated)

**Exercise 87: JIT-compile a `train_step(params, x, y, lr)` function that computes loss, gradients, and returns the updated params and loss value. Assign to `jit_train_step`.**

In [ ]:
@jax.jit
def jit_train_step(params, x, y, lr):
    loss_val, grads = jax.value_and_grad(mse_loss)(params, x, y)
    params = sgd_update(params, grads, lr)
    return params, loss_val

jit_train_step(params, X, y, 0.1)

**Exercise 88: Write a training loop that runs `jit_train_step` for 1000 steps on the XOR data with lr=0.1. Collect losses every 100 steps. Assign final params to `trained_params`.**

In [ ]:
params = init_mlp(jax.random.key(42), [2, 4, 1])
losses = []
for step in range(1000):
    params, loss_val = jit_train_step(params, X, y, 0.1)
    if step % 100 == 0:
        losses.append(float(loss_val))
        print(f"Step {step}: loss={float(loss_val):.4f}")

trained_params = params

**Exercise 89: Evaluate `trained_params` on the XOR dataset. Round the predictions to 0 or 1 and compute accuracy. Assign to `accuracy`.**

In [ ]:
preds = forward(trained_params, X)
rounded = jnp.round(jax.nn.sigmoid(preds))
accuracy = jnp.mean(rounded == y)
print(f"Predictions: {preds.flatten()}")
print(f"Rounded: {rounded.flatten()}")
print(f"Accuracy: {accuracy}")

**Exercise 90: Use `jax.vmap` to compute per-sample gradients — the gradient of the loss for each individual example in `X`, rather than the mean. Assign to `per_sample_grads`.**

In [ ]:
# Per-sample gradient: vmap over the batch dimension
def single_loss(params, x, y):
    pred = forward(params, x.reshape(1, -1))
    return jnp.mean((pred - y) ** 2)

per_sample_grads = jax.vmap(jax.grad(single_loss), in_axes=(None, 0, 0))(trained_params, X, y)
jax.tree.map(lambda x: x.shape, per_sample_grads)

<center>
    This completes Set 9: Neural Networks (Solutions 81-90) of <b>JaxTon</b>: <i>💯 JAX exercises</i>
    <br>
    You can find all the exercises and solutions on <a href="https://github.com/vopani/jaxton#exercises-">GitHub</a>
</center>